In [ ]:
import pandas as pd

# Settings
threshold = 0.0
path = r"C:\Users\TrevorWhite\Downloads\dallin_ytd.csv"

# Read & clean
df = pd.read_csv(path)
df.columns = df.columns.str.strip()

# 1) counts & totals
counts = df.groupby(['Date','Pitchtype']).size().reset_index(name='count')
totals = df.groupby('Date').size().reset_index(name='total')

# 2) usage proportions & filter
props = counts.merge(totals, on='Date')
props['prop'] = props['count'] / props['total']
props_valid = props[props['prop'] > threshold]

# 3) sum Delta_run_exp per pitch/date and per date
df_valid = df.merge(props_valid[['Date','Pitchtype']], on=['Date','Pitchtype'])
pitch_sums = (
    df_valid
    .groupby(['Date','Pitchtype'], as_index=False)['Delta_run_exp']
    .mean()
    .rename(columns={'Delta_run_exp':'Pitch_Level_RV'})
)
total_sums = (
    df_valid
    .groupby('Date', as_index=False)['Delta_run_exp']
    .mean()
    .rename(columns={'Delta_run_exp':'Total_RV'})
)

# 4) base merge (no Pitch_Level_RV here)
df_base = props_valid.merge(total_sums, on='Date')
df_base['Usage_Pct'] = (df_base['prop'] * 100).round(0).astype(int)
df_base['Avg_Total_RV_AllDates'] = df_base['Total_RV'].mean()

# 5) per-pitch averages
avg_per_pitch = (
    pitch_sums
    .groupby('Pitchtype', as_index=False)['Pitch_Level_RV']
    .mean()
    .rename(columns={'Pitch_Level_RV':'Avg_RV_Pitch_AllDates'})
)
df_full = df_base.merge(avg_per_pitch, on='Pitchtype')

# 6) pivot each pitch’s Pitch_Level_RV into its own column
rv_wide = (
    pitch_sums
    .pivot(index='Date', columns='Pitchtype', values='Pitch_Level_RV')
    .add_suffix('_RV')
    .reset_index()
)

# 7) broadcast each pitch’s average into Avg_{Pitch}_RV_AllDates
for _, row in avg_per_pitch.iterrows():
    p = row['Pitchtype']
    rv_wide[f'Avg_{p}_RV_AllDates'] = row['Avg_RV_Pitch_AllDates']

# 8) final merge & rounding
df_final = df_full.merge(rv_wide, on='Date', how='left')
cols_to_round = [c for c in df_final.columns if c.endswith('_RV') or c.startswith('Avg_')]
df_final[cols_to_round] = df_final[cols_to_round].round(2)

# 9) select & order columns
static_cols = [
    'Pitchtype','Date','count','total','Usage_Pct',
    'Total_RV','Avg_Total_RV_AllDates'
]
dynamic = [c for c in rv_wide.columns if c != 'Date']
all_cols = static_cols + sorted(dynamic)

# 10) output one table per Pitchtype
for pitch in df_final['Pitchtype'].unique():
    table = df_final[df_final['Pitchtype'] == pitch][all_cols]
    print(f"\n=== {pitch} ===")
    print(table.to_string(index=False))


In [ ]:
df_final.head()

In [ ]:
import pandas as pd
import matplotlib.pyplot as plt
import numpy as np
from scipy.stats import linregress

# assume df_final exists from previous steps
df = df_final.copy()

df.fillna(0, inplace=True)

# compute diffs
df['Diff_Total'] = df['Total_RV'] - df['Avg_Total_RV_AllDates']
rv_cols = [c for c in df.columns if c.endswith('_RV') and c != 'Total_RV']
for col in rv_cols:
    pitch_name = col[:-3]
    avg_col = f'Avg_{pitch_name}_RV_AllDates'
    df[f'Diff_{pitch_name}'] = df[col] - df[avg_col]

metric_cols = ['Diff_Total'] + [f'Diff_{c[:-3]}' for c in rv_cols]

# collect plots with outlier filter active, but skip R²/p-value filter
plots = []
for pitch in df['Pitchtype'].unique():
    df_p = df[df['Pitchtype'] == pitch]
    x_all = df_p['Usage_Pct'].to_numpy()
    cnt = df_p['count'].to_numpy()
    for metric in metric_cols:
        y_all = df_p[metric].to_numpy()
        # outlier filtering active:
        outlier_mask = (cnt < 3) & ((y_all < -0.4) | (y_all > 1))
        mask = ~outlier_mask
        
        x = x_all[mask]
        y = y_all[mask]
        if len(x) <= 1:
            continue
        
        res = linregress(x, y)
        r2 = res.rvalue**2
        pval = res.pvalue
        
        # R²/p-value filter commented out:
        if not (r2 > 10 or pval < 0.2):
            continue
        
        plots.append((pitch, metric, res, x, y))

# plot in pages of 2 cols x 3 rows
per_page = 6
for i in range(0, len(plots), per_page):
    chunk = plots[i:i+per_page]
    fig, axes = plt.subplots(3, 2, figsize=(12, 12))
    axes = axes.flatten()
    
    for ax, (pitch, metric, res, x, y) in zip(axes, chunk):
        ax.scatter(x, y, alpha=0.7)
        xx = np.linspace(x.min(), x.max(), 100)
        ax.plot(xx, res.slope * xx + res.intercept, linewidth=2, label='Trend')
        
        r2 = res.rvalue**2
        pval = res.pvalue
        ax.text(0.05, 0.95,
                f'$R^2$ = {r2:.2f}\np = {pval:.3f}',
                transform=ax.transAxes, va='top')
        
        key = metric.split('Diff_')[1]
        second = 'Total' if key == 'Total' else key
        ax.set_xlabel('Usage_Pct')
        ax.set_ylabel('Run Value Difference')
        ax.set_title(f"Usage of {pitch} vs {second} AVG Run Value")
        ax.legend()
    
    for ax in axes[len(chunk):]:
        ax.axis('off')
    
    plt.subplots_adjust(wspace=0.4, hspace=0.5, left=0.05, right=0.90, top=0.95, bottom=0.05)
    plt.show()


In [ ]:
# Read in the data files from Downloads folder
usd_pitching = pd.read_csv('C:/Users/TrevorWhite/Downloads/USDPITCHINGYTD.csv')
dallin_ytd = pd.read_csv('C:/Users/TrevorWhite/Downloads/dallin_ytd.csv')

# Rename column to match merge key
usd_pitching = usd_pitching.rename(columns={'uniqPitchId': 'Uniqpitchid'})

# Join the dataframes on uniqpitchid to get Pitchtype column
merged_df = pd.merge(usd_pitching, 
                    dallin_ytd[['Uniqpitchid', 'Pitchtype']], 
                    on='Uniqpitchid', 
                    how='left')

In [ ]:
merged_df[merged_df['pitcherAbbrevName'].str.contains('harrison', case=False, na=False)].head()

In [ ]:
# Save the merged dataframe back to the same location
merged_df.to_csv('C:/Users/TrevorWhite/Downloads/USDPITCHINGYTD.csv', index=False)


In [ ]:
merged_df['pitchTypeFull'] = merged_df['Pitchtype'].where(merged_df['Pitchtype'].notna(), merged_df['pitchTypeFull'])
merged_df = merged_df.drop('Pitchtype', axis=1)